# RawFileReaderPyAdapter — Google Colab Demo

This notebook demonstrates every public method of `RawFileAdapter` running on Google Colab.

**Steps before running:**
1. Run **Section 1** (environment setup) — installs .NET 8 and pythonnet. This takes ~60 seconds.
2. Run **Section 2** (package install) — clones the repo and installs the Python package.
3. Run **Section 3** (sample file) — upload your `.raw` file or mount Google Drive.
4. Run the remaining cells top-to-bottom.

> **Note:** After installing in Section 1 you do *not* need to restart the runtime — just continue running cells in order.

## Section 1 — Install .NET 8 Runtime

In [ ]:
%%bash
set -e

echo "=== Installing Microsoft package repository ==="
wget -q https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb -O packages-microsoft-prod.deb
dpkg -i packages-microsoft-prod.deb
rm packages-microsoft-prod.deb

echo "=== Installing .NET 8 runtime ==="
apt-get update -q
apt-get install -y dotnet-runtime-8.0 2>&1 | tail -5

echo "=== Verifying dotnet ==="
dotnet --info | head -10

## Section 2 — Install the Python Package

In [ ]:
# Clone the repo and install in editable mode so the DLLs in lib/ are on the search path.
import os, subprocess, sys

REPO_DIR = "/content/RawFileReaderPyAdapter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/mzzzhunter/RawFileReaderPyAdapter.git",
         REPO_DIR],
        check=True,
    )
    print("Cloned repo.")
else:
    print("Repo already present.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
    check=True,
)
print("Package installed.")

In [ ]:
# Set DOTNET_ROOT so pythonnet can locate the coreclr runtime.
# This MUST run before any `import rawfilereader` statement.
import os

os.environ.setdefault("DOTNET_ROOT", "/usr/share/dotnet")
print("DOTNET_ROOT =", os.environ["DOTNET_ROOT"])

## Section 3 — Provide a Sample `.raw` File

Choose **one** of the three options below, then skip the others.

### Option A — Upload from your computer

In [ ]:
from google.colab import files as _colab_files

print("Select your .raw file using the file picker below:")
uploaded = _colab_files.upload()

raw_filenames = [n for n in uploaded if n.lower().endswith(".raw")]
if not raw_filenames:
    raise ValueError("No .raw file detected in upload.")
RAW_FILE = raw_filenames[0]
print(f"RAW_FILE = {RAW_FILE!r}")

### Option B — Load from Google Drive

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# RAW_FILE = "/content/drive/MyDrive/path/to/your/sample.raw"
# print(f"RAW_FILE = {RAW_FILE!r}")

### Option C — Use the `sample.raw` bundled with the repo

The repo cloned in Section 2 includes a `sample.raw` file. Run this cell to use it directly — no upload needed.

In [ ]:
import os

REPO_DIR = "/content/RawFileReaderPyAdapter"
RAW_FILE = os.path.join(REPO_DIR, "sample.raw")

if not os.path.isfile(RAW_FILE):
    raise FileNotFoundError(
        f"{RAW_FILE} not found. Make sure Section 2 ran successfully "
        "and the repo contains sample.raw."
    )
print(f"RAW_FILE = {RAW_FILE!r}")

## Import and Verify

In [ ]:
from rawfilereader import RawFileAdapter
print("Import OK:", RawFileAdapter)

## Open file

In [ ]:
rf = RawFileAdapter(RAW_FILE)
rf.open()
print(rf)

FIRST, LAST = rf.get_scan_range()
MID = (FIRST + LAST) // 2
print(f"Scan range: {FIRST} – {LAST}  (mid={MID})")

## `is_open`

In [ ]:
print("is_open:", rf.is_open)

## `get_instrument_count()`

In [ ]:
count = rf.get_instrument_count()
print(f"Total instrument devices: {count}")

## `get_instrument_count_of_type()`

In [ ]:
for dtype in ("MS", "UV"):
    n = rf.get_instrument_count_of_type(dtype)
    print(f"  {dtype}: {n}")

## `get_instrument_type()`

In [ ]:
total = rf.get_instrument_count()
for i in range(total):
    print(f"  device[{i}]: {rf.get_instrument_type(i)}")

## `get_instrument_data()`

In [ ]:
idata = rf.get_instrument_data()
print(f"device_type    : {idata.device_type}")
print(f"name           : {idata.name}")
print(f"model          : {idata.model}")
print(f"serial_number  : {idata.serial_number}")
print(f"software_ver   : {idata.software_version}")
print(f"hardware_ver   : {idata.hardware_version}")
print(f"units          : {idata.units}")
print(f"channel_labels : {idata.channel_labels}")

## `select_instrument()`

In [ ]:
rf.select_instrument("MS", 1)
print("select_instrument('MS', 1) OK")

## `get_file_info()`

In [ ]:
fi = rf.get_file_info()
print(f"file_name              : {fi.file_name}")
print(f"creation_date          : {fi.creation_date}")
print(f"operator               : {fi.operator}")
print(f"sample_name            : {fi.sample_name}")
print(f"sample_id              : {fi.sample_id}")
print(f"sample_type            : {fi.sample_type}")
print(f"vial                   : {fi.vial}")
print(f"instrument_name        : {fi.instrument_name}")
print(f"instrument_serial_number: {fi.instrument_serial_number}")
print(f"number_of_ms_orders    : {fi.number_of_ms_orders}")
print(f"has_ms_data            : {fi.has_ms_data}")

## `get_scan_range()`

In [ ]:
first, last = rf.get_scan_range()
print(f"first={first}  last={last}  total={last - first + 1}")

## `get_start_time()` / `get_end_time()`

In [ ]:
t_start = rf.get_start_time()
t_end   = rf.get_end_time()
print(f"start_time : {t_start:.4f} min")
print(f"end_time   : {t_end:.4f} min")
print(f"run length : {t_end - t_start:.4f} min")

## `get_instrument_method()`

In [ ]:
method = rf.get_instrument_method(0)
print(f"Method length: {len(method)} chars")
print(method[:300] if method else "(empty)")

## `get_all_instrument_names_from_method()`

In [ ]:
names = rf.get_all_instrument_names_from_method()
print(f"Instrument names in method ({len(names)}):")
for n in names:
    print(f"  {n}")

## `get_filters()`

In [ ]:
filters = rf.get_filters()
print(f"Unique filters ({len(filters)}):")
for f in filters:
    print(f"  {f}")

## `get_filter_for_scan()`

In [ ]:
for scan in (FIRST, MID, LAST):
    filt = rf.get_filter_for_scan(scan)
    print(f"  scan {scan}: {filt}")

## `get_scan_event_for_scan()`

In [ ]:
ev = rf.get_scan_event_for_scan(FIRST)
print(f"Scan event for scan {FIRST}:")
for k, v in ev.items():
    print(f"  {k}: {v}")

## `get_scan_stats()`

In [ ]:
stats = rf.get_scan_stats(FIRST)
print(f"scan_number        : {stats.scan_number}")
print(f"start_time         : {stats.start_time:.4f} min")
print(f"tic                : {stats.tic:.3e}")
print(f"base_peak_mass     : {stats.base_peak_mass:.4f}")
print(f"base_peak_intensity: {stats.base_peak_intensity:.3e}")
print(f"low_mass           : {stats.low_mass:.2f}")
print(f"high_mass          : {stats.high_mass:.2f}")
print(f"packet_count       : {stats.packet_count}")
print(f"is_centroid_scan   : {stats.is_centroid_scan}")

## `get_retention_time()`

In [ ]:
for scan in (FIRST, MID, LAST):
    rt = rf.get_retention_time(scan)
    print(f"  scan {scan}: {rt:.4f} min")

## `scan_number_from_retention_time()`

In [ ]:
mid_rt = rf.get_retention_time(MID)
scan_back = rf.scan_number_from_retention_time(mid_rt)
print(f"RT {mid_rt:.4f} min → scan {scan_back}  (mid was {MID})")

## `get_centroid_stream()`

In [ ]:
centroid = rf.get_centroid_stream(FIRST)
print(f"scan_number  : {centroid.scan_number}")
print(f"peaks        : {len(centroid.masses)}")
print(f"is_exceptional: {centroid.is_exceptional}")
print(f"is_reference  : {centroid.is_reference}")
print()
print("First 5 (mass, intensity) peaks:")
for mass, intensity in centroid.peaks[:5]:
    print(f"  m/z={mass:.4f}  I={intensity:.2e}")

## `get_profile_data()`

In [ ]:
profile = rf.get_profile_data(FIRST)
print(f"scan_number : {profile.scan_number}")
print(f"segments    : {len(profile.segments)}")
masses = profile.masses
intensities = profile.intensities
print(f"data points : {len(masses)}")
if masses:
    print(f"m/z range   : {masses[0]:.2f} – {masses[-1]:.2f}")
print("First 5 (mass, intensity):")
for m, i in zip(masses[:5], intensities[:5]):
    print(f"  m/z={m:.4f}  I={i:.2e}")

## `get_scan_info()`

In [ ]:
info = rf.get_scan_info(FIRST)
print(f"scan_number     : {info.scan_number}")
print(f"ms_order        : {info.ms_order}")
print(f"retention_time  : {info.retention_time:.4f} min")
print(f"scan_filter     : {info.scan_filter}")
print(f"is_centroid     : {info.is_centroid}")
print(f"detector_type   : {info.detector_type}")
print(f"injection_time  : {info.injection_time:.2f} ms")
print(f"precursor_mass  : {info.precursor_mass}")
print(f"precursor_charge: {info.precursor_charge}")
print(f"collision_energy: {info.collision_energy}")
print(f"isolation_width : {info.isolation_width}")

## `average_scans_in_range()`

In [ ]:
end = min(FIRST + 9, LAST)
avg = rf.average_scans_in_range(FIRST, end)
print(f"first_scan : {avg.first_scan}")
print(f"last_scan  : {avg.last_scan}")
print(f"peaks      : {len(avg.masses)}")
print("First 5 averaged peaks:")
for m, i in zip(avg.masses[:5], avg.intensities[:5]):
    print(f"  m/z={m:.4f}  I={i:.2e}")

## `average_scans()`

In [ ]:
scan_list = list(range(FIRST, min(FIRST + 5, LAST + 1)))
avg2 = rf.average_scans(scan_list)
print(f"averaged scans : {scan_list}")
print(f"first_scan     : {avg2.first_scan}")
print(f"last_scan      : {avg2.last_scan}")
print(f"peaks          : {len(avg2.masses)}")

## `get_chromatogram()` — TIC

In [ ]:
tic = rf.get_chromatogram(trace_type="TIC")
print(f"trace_type : {tic.trace_type}")
print(f"points     : {len(tic.times)}")
print("First 5 TIC points:")
for t, i in zip(tic.times[:5], tic.intensities[:5]):
    print(f"  {t:.4f} min  {i:.3e}")

## `get_chromatogram()` — BasePeak

In [ ]:
bpc = rf.get_chromatogram(trace_type="BasePeak")
print(f"trace_type : {bpc.trace_type}")
print(f"points     : {len(bpc.times)}")
print("First 5 BasePeak points:")
for t, i in zip(bpc.times[:5], bpc.intensities[:5]):
    print(f"  {t:.4f} min  {i:.3e}")

## `get_chromatogram()` — MassRange (EIC)

In [ ]:
bp_mass = rf.get_scan_stats(FIRST).base_peak_mass
mass_range_str = f"{bp_mass - 0.5:.3f}-{bp_mass + 0.5:.3f}"
eic = rf.get_chromatogram(trace_type="MassRange", mass_range=mass_range_str)
print(f"trace_type : {eic.trace_type}")
print(f"mass_range : {eic.mass_range}")
print(f"points     : {len(eic.times)}")
print("First 5 EIC points:")
for t, i in zip(eic.times[:5], eic.intensities[:5]):
    print(f"  {t:.4f} min  {i:.3e}")

## `get_trailer_data()`

In [ ]:
trailer = rf.get_trailer_data(FIRST)
print(f"scan_number : {trailer.scan_number}")
print(f"fields ({len(trailer.fields)}):")
for k, v in list(trailer.fields.items())[:10]:
    print(f"  {k!r}: {v!r}")

## `get_trailer_header_info()`

In [ ]:
headers = rf.get_trailer_header_info()
print(f"Trailer labels ({len(headers)}):")
for h in headers[:10]:
    print(f"  {h}")

## `get_status_log_header_info()`

In [ ]:
log_headers = rf.get_status_log_header_info()
print(f"Status log labels ({len(log_headers)}):")
for h in log_headers[:10]:
    print(f"  {h}")

## `get_status_log_for_retention_time()`

In [ ]:
rt_query = rf.get_retention_time(FIRST)
log_entry = rf.get_status_log_for_retention_time(rt_query)
print(f"retention_time : {log_entry.retention_time:.4f} min")
print(f"fields ({len(log_entry.fields)}):")
for k, v in list(log_entry.fields.items())[:10]:
    print(f"  {k!r}: {v!r}")

## `get_status_log_for_scan()`

In [ ]:
log_scan = rf.get_status_log_for_scan(FIRST)
print(f"retention_time : {log_scan.retention_time:.4f} min")
print(f"fields         : {len(log_scan.fields)}")

## `get_scan_dependents()`

In [ ]:
dep = rf.get_scan_dependents(FIRST, depth=1)
print(f"scan_number           : {dep.scan_number}")
print(f"dependent_scan_numbers: {dep.dependent_scan_numbers}")

## `iter_scan_info()`

In [ ]:
ms1_infos = [info for info in rf.iter_scan_info(ms_order=1)]
print(f"MS1 scans found: {len(ms1_infos)}")
print("First 3:")
for info in ms1_infos[:3]:
    print(f"  scan={info.scan_number}  rt={info.retention_time:.3f} min  filter={info.scan_filter[:60]}")

## `iter_centroid_data()`

In [ ]:
ms1_spectra = list(rf.iter_centroid_data(ms_order=1))
print(f"MS1 centroid spectra collected: {len(ms1_spectra)}")
print("First 3 (scan, peak_count):")
for cd in ms1_spectra[:3]:
    print(f"  scan={cd.scan_number}  peaks={len(cd.masses)}")

## `analyze_all_scans()`

In [ ]:
summary = rf.analyze_all_scans()
print(f"total       : {summary['total']}")
print(f"centroid    : {summary['centroid']}")
print(f"profile     : {summary['profile']}")
print(f"out_of_order: {summary['out_of_order']}")
if summary['out_of_order'] == 0:
    print("All scans passed mass-ordering check.")
else:
    print(f"WARNING: {summary['out_of_order']} scan(s) have out-of-order masses.")

## `subtract_spectra()`

In [ ]:
from collections import defaultdict

filter_to_scans = defaultdict(list)
for scan in range(FIRST, LAST + 1):
    filt = rf.get_filter_for_scan(scan)
    filter_to_scans[filt].append(scan)
    if any(len(v) >= 2 for v in filter_to_scans.values()):
        break

pair = next((v for v in filter_to_scans.values() if len(v) >= 2), None)
if pair is None:
    print("No two scans share the same filter — skipping subtract_spectra test")
else:
    scan_a, scan_b = pair[0], pair[1]
    result = rf.subtract_spectra(scan_a, scan_b)
    print(f"scan_a     : {result.scan_a}")
    print(f"scan_b     : {result.scan_b}")
    print(f"scan_filter: {result.scan_filter}")
    print(f"is_centroid: {result.is_centroid}")
    print(f"peaks total: {len(result.masses)}")
    print(f"peaks (+)  : {len(result.peaks)}")
    print("First 5 signed (mass, intensity):")
    for m, i in zip(result.masses[:5], result.intensities[:5]):
        print(f"  m/z={m:.4f}  {i:+.3e}")

## `subtract_background()` *(requires BackgroundSubtraction.dll)*

In [ ]:
from rawfilereader.exceptions import AssemblyLoadError

bg_scans = list(range(FIRST, min(FIRST + 3, LAST + 1)))
signal_scan = min(FIRST + 10, LAST)
try:
    bg_result = rf.subtract_background(
        scan_number=signal_scan,
        background_scan_numbers=bg_scans,
    )
    print(f"scan_number     : {bg_result.scan_number}")
    print(f"background_scans: {bg_result.background_scans}")
    print(f"scan_filter     : {bg_result.scan_filter}")
    print(f"peaks           : {len(bg_result.peaks)}")
    print("First 5 peaks:")
    for m, i in bg_result.peaks[:5]:
        print(f"  m/z={m:.4f}  I={i:.3e}")
except AssemblyLoadError as e:
    print(f"BackgroundSubtraction.dll not available: {e}")

## Close file

In [ ]:
rf.close()
print("is_open() after close:", rf.is_open)

---

## Bonus — Context Manager (`with` statement)

`RawFileAdapter` implements `__enter__` / `__exit__`, so the file is closed automatically even if an error occurs mid-block. This is the recommended pattern for scripts and production code.

In [ ]:
from rawfilereader import RawFileAdapter

with RawFileAdapter(RAW_FILE) as rf:
    first, last = rf.get_scan_range()
    mid = (first + last) // 2

    fi = rf.get_file_info()
    print(f"File       : {fi.file_name}")
    print(f"Instrument : {fi.instrument_name}")
    print(f"Scans      : {first} – {last}  (total {last - first + 1})")

    tic = rf.get_chromatogram(trace_type="TIC")
    print(f"TIC points : {len(tic.times)}")

    centroid = rf.get_centroid_stream(mid)
    print(f"Scan {mid} centroid peaks: {len(centroid.masses)}")

    stats = rf.get_scan_stats(mid)
    print(f"Scan {mid} base peak: m/z={stats.base_peak_mass:.4f}  I={stats.base_peak_intensity:.3e}")

# File is now closed automatically — no rf.close() needed.
print(f"\nis_open after 'with' block: {rf.is_open}")